# Access control, least privilege and secret handling

**Module 03 — Cybersecurity Fundamentals for AI Systems**

Every AI system inherits classical security problems: who may do what, with which credential, and what
happens when that credential leaks. This notebook implements a deny-by-default policy engine, measures
each role's authority, and demonstrates the secret-handling mistakes that show up in real ML codebases.

## Learning objectives

1. Encode roles, permissions and resources as data and enforce them with a deny-by-default engine.
2. Compute each role's effective authority and find the over-privileged one.
3. Explain why scoping a token to a resource is stronger than checking a role name.
4. Hash a password with a salt and a slow KDF, and compare secrets in constant time.
5. Redact secrets from logs and detect the leaks that remain.

## Background

Three ideas from Module 03 do most of the work in practice:

* **Complete mediation** — every access is checked, not just the first one.
* **Least privilege** — a component gets the authority its task needs and no more.
* **Deny by default** — the policy engine says no unless a rule says yes.

For models these translate into concrete questions: may this training job read the raw table or only the
de-identified view? May this inference service write to the audit log but not read it? Is the provider key
reachable from the notebook kernel?

## Dependencies and installation

```bash
python -m pip install "pandas>=2.2"
```

Standard library only (`hashlib`, `hmac`, `secrets`, `re`). CPU, sub-second runtime, no network.

In [1]:
import hashlib
import hmac
import re
import secrets
from dataclasses import dataclass, field

import pandas as pd

print("ready")

ready


**Expected output** — `ready`. No global state is created here; every demo below is self-contained.

### Step 1 — Roles, resources and actions as data

The policy is a set of explicit grants. Anything not granted is denied, which is what "deny by default"
means in code rather than in a policy document.

In [2]:
ROLES = {
    "data_engineer": {"dataset:raw": {"read", "write"}, "dataset:curated": {"read"}},
    "ml_engineer": {"dataset:curated": {"read"}, "model:artefact": {"read", "write"}},
    "service": {"model:artefact": {"read"}, "audit:log": {"write"}, "patient:record": {"read"}},
    "auditor": {"audit:log": {"read"}},
}


def authorise(role: str, resource: str, action: str, subject: str | None = None) -> tuple[bool, str]:
    # Deny-by-default check. Returns (allowed, reason) so the caller can log a decision.
    if role not in ROLES:
        return False, f"unknown role {role!r}"
    grants = ROLES[role].get(resource)
    if not grants:
        return False, f"{role} has no grant on {resource}"
    if action not in grants:
        return False, f"{role} may not {action} on {resource}"
    return True, "allowed"


tests = [
    ("service", "patient:record", "read"),
    ("service", "patient:record", "write"),
    ("auditor", "audit:log", "read"),
    ("auditor", "audit:log", "write"),
    ("unknown", "dataset:raw", "read"),
]
for role, res, act in tests:
    ok, why = authorise(role, res, act)
    print(f"{role:14} {act:6} {res:15} -> {ok!s:5} ({why})")

service        read   patient:record  -> True  (allowed)
service        write  patient:record  -> False (service may not write on patient:record)
auditor        read   audit:log       -> True  (allowed)
auditor        write  audit:log       -> False (auditor may not write on audit:log)
unknown        read   dataset:raw     -> False (unknown role 'unknown')


**Expected output** — reads allowed only where granted; `service` cannot write patient records;
`auditor` cannot write the log it audits; the unknown role is denied rather than defaulted. Each denial
carries a reason, which is what makes the audit log useful later (Module 18).

### Step 2 — Measure effective authority

"Least privilege" is a claim you can quantify: count the (resource, action) pairs each role can perform.
The role with the largest count is the one to justify in review.

In [3]:
rows = []
for role, grants in ROLES.items():
    pairs = {(res, act) for res, acts in grants.items() for act in acts}
    rows.append({"role": role, "resources": len(grants), "permissions": len(pairs),
                 "writes": sum(1 for _, a in pairs if a == "write")})
authority = pd.DataFrame(rows).sort_values("permissions", ascending=False)
authority

,role,resources,permissions,writes
0,data_engineer,2,3,1
1,ml_engineer,2,3,1
2,service,3,3,1
3,auditor,1,1,0


**Expected output** — a table where `data_engineer` and `ml_engineer` hold the most permissions and
`service` holds the most *sensitive* ones. The count is crude, but it converts an argument about
privilege into a number two reviewers can disagree about specifically.

### Step 3 — Scope the credential, not just the role

A role check inside the service is necessary and insufficient: if the database credential can read every
tenant, one bug is a breach. Below, the same logical request is evaluated two ways — by role only, and by
role **plus** a resource-scoped token.

In [4]:
@dataclass
class Session:
    role: str
    tenant: str
    scopes: set[str] = field(default_factory=set)


def read_record(session: Session, record: dict) -> tuple[bool, str]:
    ok, why = authorise(session.role, "patient:record", "read")
    if not ok:
        return False, why
    if record["tenant"] != session.tenant:
        return False, "record outside token scope"
    if "records:read" not in session.scopes:
        return False, "token lacks records:read scope"
    return True, "allowed"


record = {"id": "P-1", "tenant": "acme"}
role_only = Session("service", tenant="acme", scopes=set())
scoped = Session("service", tenant="acme", scopes={"records:read"})
wrong_tenant = Session("service", tenant="globex", scopes={"records:read"})

for label, sess in [("role only", role_only), ("scoped", scoped), ("wrong tenant", wrong_tenant)]:
    print(f"{label:13} -> {read_record(sess, record)}")

role only     -> (False, 'token lacks records:read scope')
scoped        -> (True, 'allowed')
wrong tenant  -> (False, 'record outside token scope')


**Expected output** — only `scoped` succeeds; `wrong tenant` is refused even though the role and scope
are valid, because the *resource* is bound to the session. This is the confused-deputy defence Module 14
revisits for agents.

### Step 4 — Secrets: slow hashing for passwords, constant-time comparison

Two rules, both easy to get wrong. Passwords are stored as salted, iterated hashes — never encrypted,
never plain. Secret comparison uses a constant-time function so that comparison time does not leak how
many leading bytes matched.

In [5]:
def hash_password(password: str, salt: bytes | None = None, iterations: int = 200_000) -> tuple[bytes, bytes]:
    salt = salt or secrets.token_bytes(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)
    return salt, digest


def verify_password(password: str, salt: bytes, expected: bytes, iterations: int = 200_000) -> bool:
    candidate = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)
    return hmac.compare_digest(candidate, expected)


salt, stored = hash_password("correct horse battery staple")
print("salt length:", len(salt), "hash length:", len(stored))
print("correct password:", verify_password("correct horse battery staple", salt, stored))
print("wrong password  :", verify_password("correct horse battery stapl", salt, stored))

salt2, stored2 = hash_password("correct horse battery staple")
print("same password, different salt gives different hash:", stored != stored2)

salt length: 16 hash length: 32


correct password: True


wrong password  : False


same password, different salt gives different hash: True


**Expected output** — verification true only for the exact password, and two hashes of the *same*
password that differ because the salts differ. Identical hashes across users are how an attacker spots a
shared password without cracking anything.

### Step 5 — Redact before logging, then check what leaked

Logs are the most common leak path for credentials. A redaction function applied to every log line is
cheaper than auditing every call site; the check below shows both what it catches and what it misses.

In [6]:
SECRET_PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9]{8,}"), "[REDACTED_API_KEY]"),
    (re.compile(r"(?i)password=\S+"), "password=[REDACTED]"),
    (re.compile(r"Bearer\s+[A-Za-z0-9._-]+"), "Bearer [REDACTED]"),
]


def redact(line: str) -> str:
    for pattern, replacement in SECRET_PATTERNS:
        line = pattern.sub(replacement, line)
    return line


samples = [
    "provider call with sk-live-9f2b7c41a0 done",
    "login attempt password=hunter2 from 10.0.0.7",
    "header Authorization: Bearer eyJhbGciOi.abc.def",
    "provider key stored in env var PROVIDER_KEY",
]
for line in samples:
    cleaned = redact(line)
    print(("LEAK  " if any(p.search(cleaned) for p, _ in SECRET_PATTERNS) else "clean ") + cleaned)

clean provider call with sk-live-9f2b7c41a0 done
LEAK  login attempt password=[REDACTED] from 10.0.0.7
clean header Authorization: Bearer [REDACTED]
clean provider key stored in env var PROVIDER_KEY


**Expected output** — three lines redacted, and the fourth printed `clean` while still telling you a key
exists in an environment variable. That fourth line is the honest limitation: pattern matching catches
shapes, not meaning. Module 16 and 18 add structured logging and secret scanning to close it.

## Exercises

1. **Add a tenant-scoped role.** Introduce `tenant_admin` that may read and write records only within its
   own tenant, and extend `authorise` to enforce it without breaking existing calls.
2. **Break complete mediation.** Remove the scope check in `read_record` and show which of the three
   sessions now succeeds. Explain the resulting bug in one sentence.
3. **Tune the KDF.** Time `hash_password` at 10 000, 100 000 and 200 000 iterations. Which would you ship
   for an interactive login, and what does the answer depend on?
4. **Attack your own redactor.** Write a log line containing a secret in a shape the patterns miss, then
   add a pattern for it. What class of secret can never be caught this way?

## Reflection questions

1. Why is a role check inside the application weaker than a scoped credential, even when the check is
   correct?
2. The policy table grants `service` write access to the audit log. What can go wrong, and what would you
   add to the design?
3. Which of the three CIA properties does password hashing protect, and which does it not?

## References

1. Saltzer, J. and Schroeder, M., "The Protection of Information in Computer Systems", 1975.
2. NIST, *SP 800-63B*, Digital Identity Guidelines — password storage requirements.
3. OWASP, *Application Security Verification Standard*, v4.0.3 — authentication and access control.
4. NIST, *SP 800-53 Rev. 5*, AC and IA control families.